# Semana 6 — Circuitos Cuánticos con Qiskit
## De las matrices a los circuitos reales

**Objetivo:** Usar Qiskit para construir, simular y visualizar circuitos cuánticos. Pasar del modelo matemático de las semanas anteriores a la herramienta estándar de la industria.

**Hito verificable:** Puedes construir cualquier circuito visto hasta ahora en Qiskit, simularlo con Aer y obtener histogramas de resultados.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from qiskit.quantum_info import Statevector, DensityMatrix
import numpy as np
import matplotlib.pyplot as plt
print('Qiskit importado correctamente')

## Ejercicio 6.1 — Primer circuito: superposición y medición

In [ ]:
# Circuito de 1 qubit: H luego medir
qc = QuantumCircuit(1, 1)  # 1 qubit cuántico, 1 bit clásico
qc.h(0)     # Hadamard en qubit 0
qc.measure(0, 0)  # medir qubit 0 → bit clásico 0

print('Circuito:')
print(qc.draw(output='text'))

# Simular con AerSimulator
sim = AerSimulator()
job = sim.run(qc, shots=10000)
counts = job.result().get_counts()
print(f'\nConteos (10 000 shots): {counts}')
print(f'Frecuencia |0⟩: {counts.get("0", 0)/10000:.4f} (esperado: 0.5)')
print(f'Frecuencia |1⟩: {counts.get("1", 0)/10000:.4f} (esperado: 0.5)')

## Ejercicio 6.2 — Estado de Bell en Qiskit

In [ ]:
# Crear |Φ+⟩ = (|00⟩ + |11⟩)/√2
qc_bell = QuantumCircuit(2, 2)
qc_bell.h(0)       # H en qubit 0
qc_bell.cx(0, 1)   # CNOT: control=0, target=1
qc_bell.measure([0, 1], [0, 1])

print('Circuito de Bell:')
print(qc_bell.draw(output='text'))

job_bell = sim.run(qc_bell, shots=10000)
counts_bell = job_bell.result().get_counts()
print(f'\nConteos: {counts_bell}')
print('→ Solo |00⟩ y |11⟩ aparecen. Nunca |01⟩ o |10⟩ — están correlacionados.')

## Ejercicio 6.3 — Vector de estado con Qiskit

In [ ]:
# Obtener el statevector SIN medir
qc_sv = QuantumCircuit(2)
qc_sv.h(0)
qc_sv.cx(0, 1)

sv = Statevector(qc_sv)
print('Statevector de |Φ+⟩:')
print(sv)
print()
print('Probabilidades:')
for estado, prob in sv.probabilities_dict().items():
    if prob > 0.001:
        print(f'  |{estado}⟩: {prob:.4f}')

# Circuito más complejo: algoritmo de Deutsch
qc_deutsch = QuantumCircuit(2)
qc_deutsch.x(1)      # Preparar |01⟩
qc_deutsch.h(0)
qc_deutsch.h(1)
qc_deutsch.cx(0, 1)  # Oráculo f(x)=x (CNOT)
qc_deutsch.h(0)

sv_deutsch = Statevector(qc_deutsch)
print('\nEstado tras el algoritmo de Deutsch (f=identidad):')
print(sv_deutsch)
print('→ El qubit 0 está en |1⟩: f es balanceada ✓')

## Ejercicio 6.4 — Circuito de 3 qubits: superposición uniforme

In [ ]:
# Superposición uniforme sobre todos los 2^n estados base
n = 3
qc_sup = QuantumCircuit(n, n)
qc_sup.h(range(n))  # H en todos los qubits
qc_sup.measure(range(n), range(n))

print(f'Circuito de superposición uniforme ({n} qubits):')
print(qc_sup.draw(output='text'))

job_sup = sim.run(qc_sup, shots=8000)
counts_sup = job_sup.result().get_counts()

print(f'\nEsperado: cada estado con probabilidad 1/8 = {1/8:.4f}')
print(f'Resultados (ordenados):')
for estado in sorted(counts_sup.keys()):
    n_mediciones = counts_sup[estado]
    freq = n_mediciones / 8000
    barra = '█' * int(freq * 80)
    print(f'  |{estado}⟩: {n_mediciones:4d}  ({freq:.4f})  {barra}')

## Ejercicio 6.5 — Transpilación y puertas nativas del hardware

In [ ]:
from qiskit.compiler import transpile

# El hardware real no entiende todas las puertas — solo un conjunto nativo
# Qiskit transpila el circuito al conjunto de puertas del dispositivo
qc_complejo = QuantumCircuit(2)
qc_complejo.h(0)
qc_complejo.t(0)
qc_complejo.cx(0, 1)
qc_complejo.ry(np.pi/3, 1)
qc_complejo.s(0)

# Transpilar para un conjunto de puertas básicas
qc_transpilado = transpile(qc_complejo, basis_gates=['cx', 'rz', 'sx', 'x'],
                           optimization_level=2)

print('Circuito original:')
print(qc_complejo.draw(output='text'))
print(f'Puertas originales: {qc_complejo.count_ops()}')
print()
print('Circuito transpilado (base: cx, rz, sx, x):')
print(qc_transpilado.draw(output='text'))
print(f'Puertas transpiladas: {qc_transpilado.count_ops()}')

# Verificar equivalencia
sv_original = Statevector(qc_complejo)
sv_transpilado = Statevector(qc_transpilado)
equiv = sv_original.equiv(sv_transpilado)
print(f'\n¿Son equivalentes (hasta fase global)? {equiv}')

## Mini reto — Completar antes de avanzar a la Semana 7

Construye en Qiskit el circuito de teleportación cuántica que implementaste matemáticamente en la Semana 5:
1. Alice tiene |ψ⟩ = Ry(π/3)|0⟩
2. Usa medición condicionada (instrucciones `if_test` o `measure_all`)
3. Verifica que el statevector en el qubit de Bob coincide con el estado original

Pista: en Qiskit puedes usar `qc.if_test((clbit, 1)):` para correcciones condicionadas.

In [ ]:
# TU CÓDIGO AQUÍ
def teleportacion_qiskit(theta: float) -> QuantumCircuit:
    """Construye el circuito de teleportación en Qiskit."""
    # TODO: implementar
    pass

## ✅ Hito de la Semana 6

Antes de avanzar, comprueba que puedes:

- [ ] Crear circuitos con QuantumCircuit y ejecutarlos con AerSimulator
- [ ] Obtener y analizar el statevector sin medir
- [ ] Construir el estado de Bell y verificar las correlaciones
- [ ] Transpilar un circuito a un conjunto de puertas nativas
- [ ] Leer e interpretar histogramas de resultados

## Reflexión (escribe aquí tu respuesta)

**¿Por qué es necesario transpilar los circuitos para hardware real?**

*(tu respuesta aquí)*

**¿Qué limitación introduce el ruido del hardware real que el simulador no tiene?**

*(tu respuesta aquí)*

---
*Siguiente: Semana 7 — El Algoritmo de Grover*